In [11]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

import matplotlib.cm as cm
import colormaps as cmaps
import matplotlib.colors as colors
from labellines import labelLines
from matplotlib.cm import brg, ScalarMappable
from matplotlib.colors import (
    LogNorm,
    TwoSlopeNorm,
    LinearSegmentedColormap,
    Normalize,
    ListedColormap,
)
from matplotlib.lines import Line2D

import jax
import jax.numpy as jnp
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import yaml
from pathlib import Path

import EI.ei_unified as eu
import EI.ei_jax as ej
from EI.ei_utils import gap_info
import EI.eff_T as et

from utils.logger import logger, tqdm_bar
from utils.plotting_utils import plot_cmap, plot_sweep_panels, plot_single, plot_multi, SweepColor

from utils.cosmetics import apply_plt_style, SET1_LIST
apply_plt_style()

OUT = "/home/kzeleznikar/IJS-F1/Koda/EI_baths/"
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("png", dpi=300)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from utils.plotting_utils import Plotter

pt = Plotter()

In [13]:
cfg_path = Path("/home/kzeleznikar/IJS-F1/Koda/EI_baths/config/config_test.yaml")

with cfg_path.open("r") as f:
    cfg = yaml.safe_load(f)

md = cfg["model"]
eq = cfg["equilibrium"]
sc = cfg["scan"]
op = cfg["open_system"]

nk = md["nk"]
bp = md["band"]["pars"]

bd = eu.tb_2d(nk["scan"], **bp)
bd_ref = eu.tb_2d(nk["reference"], **bp)

p = eu.MFPars(**md["mean_field"])

In [14]:
s0 = eu.solve_eq(bd_ref, p, **eq["initial"], **eq["solve"])

d0 = abs(s0.d)
m0 = s0.m
mu0 = s0.mu

# za normalizacijo
tc = eu.critical_temperature(bd_ref, p, **eq["critical"])

logger.info(f"Delta_0 = {d0}")
logger.info(f"m_0     = {m0}")
logger.info(f"mu0     = {mu0}")
logger.info(f"T_c     = {tc}")

kriticna temp bisekcija:   0%|          | 0/100 [00:00<?, ?it/s]

[11:36:22] [INFO    ] Delta_0 = 1.0000516576539338
[11:36:22] [INFO    ] m_0     = -0.6843383516761253
[11:36:22] [INFO    ] mu0     = 1.230852907732626
[11:36:22] [INFO    ] T_c     = 0.5352354225531222


In [15]:
tg = sc["temperature"]

tn = np.linspace( tg["min_ratio"], tg["max_ratio"], tg["nt"])

t1a = tn * tc
t2a = tn * tc
nt = tg["nt"]

In [16]:
bc = op["bath"]

rf = getattr(ej, bc["rate"])

bpars = dict(bc["pars"])
bpars["rate"] = rf

set mu_B = E_c

In [17]:
Ec = md["mean_field"]["v"] * md["mean_field"]["n"] /2 + (md["band"]["pars"]["ta"] + md["band"]["pars"]["tb"]) / (2 * (md["band"]["pars"]["ta"] - md["band"]["pars"]["tb"])) * (md["mean_field"]["v"] * s0.m - md["band"]["pars"]["gap"])
logger.info(f"E_c = {Ec}")

bpars["mu"] = Ec

[11:36:22] [INFO    ] E_c = 2.2


Phase diagrams, different gaps

In [ ]:
da = np.full((nt, nt), np.nan)
ma = np.full((nt, nt), np.nan)
er = np.full((nt, nt), np.nan)
it = np.zeros((nt, nt), dtype=int)
ok = np.zeros((nt, nt), dtype=bool)

gh = np.full((nt, nt), np.nan)
gg = np.full((nt, nt), np.nan)
gd = np.full((nt, nt), np.nan)
gi = np.full((nt, nt), np.nan)


de = s0.d
me = s0.m
ds = 0.2 * max(d0, 1.0)

for j in tqdm(range(nt), desc="T2"):
    t2 = t2a[j]

    se = eu.solve_eq(
        bd, p, t=t2,
        d=max(abs(de), 1.0e-8),
        m=me,
        mix=(0.15, 0.15),
        tol=1.0e-9,
        nmax=20000,
        chk=20,
        prog=False,
    )

    n = se.n.copy()
    d = se.d
    m = se.m

    for i in tqdm(
        range(j, nt),
        desc="T1",
        leave=False,
    ):
        t1 = t1a[i]

        bs = (
            ej.gam_db(t=t1, name="bath 1", **bpars),
            ej.gam_db(t=t2, name="bath 2", **bpars),
        )

        so = ej.solve_open(
            bd, p, n, bs,
            d=d, m=m,
            **op["solve"],
        )

        ee = np.asarray(so.st.e)
        
        da[j, i] = abs(so.d)
        ma[j, i] = so.m
        er[j, i] = so.err
        it[j, i] = so.it
        ok[j, i] = so.ok

        gp = gap_info(bd, so.st)

        gh[j, i] = gp["hartree_g"]
        gg[j, i] = gp["diag_g"]
        gd[j, i] = gp["diag_min"]
        gi[j, i] = gp["diag_ind"]

        n = so.n.copy()
        d = max(abs(so.d), ds)
        m = so.m

        da[i, j] = da[j, i]
        ma[i, j] = ma[j, i]
        er[i, j] = er[j, i]
        it[i, j] = it[j, i]
        ok[i, j] = ok[j, i]

        gh[i, j] = gh[j, i]
        gg[i, j] = gg[j, i]
        gd[i, j] = gd[j, i]
        gi[i, j] = gi[j, i]

    de = se.d
    me = se.m

In [ ]:
plot_cmap(tn, tn, da / d0, r"$G^{\mathrm{H}}(\Gamma)/\Delta_0$", cmap=cmaps.lipari)
plot_cmap(tn, tn, gh / d0, r"$G^{\mathrm{H}}(\Gamma) / \Delta_0$", cmaps.bubblegum)
plot_cmap(tn, tn, gg / d0, r"$G^{\mathrm{diag}}(\Gamma) / \Delta_0$", cmaps.batlow,)
plot_cmap(tn, tn, gi / (2 * d0), r"$G_{\mathrm{ind}}^{\mathrm{diag}} / 2 \Delta_0$", cmaps.amethyst,)
plot_cmap(tn, tn, ma / (2 * d0), r"$(n_a-n_b) / 2 \Delta_0$", cmaps.gem)

1D sweep of the phase diagrams

In [ ]:
va = np.linspace(4.4, 7.8, 7)
xa = np.linspace(0.01, 100.0, 20*nt)

nv = va.size
nx = xa.size
sh = (nv, nx)

dr = np.full(sh, np.nan)
hr = np.full(sh, np.nan)
mr = np.full(sh, np.nan)
ir = np.full(sh, np.nan)
la = np.full(sh, np.nan)

era = np.full(sh, np.nan)
oka = np.zeros(sh, dtype=bool)

dza = np.full(nv, np.nan)
tca = np.full(nv, np.nan)
nf = float(md["mean_field"]["n"])
uh0 = md["mean_field"]["uh"]

for r, vv in tqdm(
    enumerate(va),
    total=nv,
    desc="V",
):
    pv = eu.MFPars(
        v=vv,
        n=nf,
        uh=uh0,
    )

    # Zero-temperature equilibrium normalization
    sz = eu.solve_eq(
        bd,
        pv,
        t=0.0,
        d=eq["initial"]["d"],
        m=eq["initial"]["m"],
        **eq["solve"],
    )

    dz = abs(sz.d)
    dza[r] = dz

    # Critical temperature for this V
    cp = {
        **eq["critical"],
        "prog": False,
    }
    tv = eu.critical_temperature(
        bd,
        pv,
        **cp,
    )
    tca[r] = tv

    t2 = 0.01 * tv

    # Equilibrium state used to initialize the cold bath
    se = eu.solve_eq(
        bd,
        pv,
        t=t2,
        d=max(dz, 1.0e-8),
        m=sz.m,
        **eq["solve"],
    )

    nq = se.n.copy()
    d = se.d
    m = se.m
    ds = 0.2 * max(dz, 1.0)

    bpars["mu"] = vv / 2.0
    
    for i, x1 in tqdm(
        enumerate(xa),
        total=nx,
        desc="T1",
        leave=False,
    ):
        t1 = x1 * tv

        bs = (
            ej.gam_db(
                t=t1,
                name="bath 1",
                **bpars,
            ),
            ej.gam_db(
                t=t2,
                name="bath 2",
                **bpars,
            ),
        )

        so = ej.solve_open(
            bd,
            pv,
            nq,
            bs,
            d=d,
            m=m,
            **op["solve"],
        )

        ee = np.asarray(so.st.e)
        nn = np.asarray(so.n)

        de = ee[0] - ee[1]
        la[r, i] = vv * np.sum(
            bd.w
            * (nn[1] - nn[0])
            / np.maximum(de, 1.0e-12))

        gp = gap_info(bd, so.st)

        dr[r, i] = abs(so.d) / dz
        hr[r, i] = gp["hartree_g"] / dz
        mr[r, i] = so.m / (2.0 * dz)
        ir[r, i] = gp["diag_ind"] / (2.0 * dz)

        era[r, i] = so.err
        oka[r, i] = so.ok

        nq = so.n.copy()
        d = max(abs(so.d), ds)
        m = so.m

In [ ]:
ys = [dr, hr, mr, ir]
yl = [
    r"$\Delta/\Delta_0$",
    r"$G_\Gamma^{\mathrm{H}}/\Delta_0$",
    r"$(n_a-n_b)/(2\Delta_0)$",
    r"$G_{\mathrm{ind}}^{\mathrm{diag}}/(2\Delta_0)$",
]

plot_sweep_panels(
    xa,
    ys,
    va,
    yl,
    cmap=cmaps.guppy,
    refs=[None, 0.0, None, None],
)

In [ ]:
sw = SweepColor(va, cmaps.guppy, label="$V$")
plot_single(xa, la, r"$T_1/T_c$", r"$\Lambda$", pt=pt, sw=sw,  name="lambda_plot")

Dispersion bands, and occupation distributions for 5 different temperature ratios. 

In [18]:
vv = 7.8
xp = np.array([0.1, 1.1, 2.0, 4.0, 6.0])

# Include the requested points exactly in the continuation grid
nx = max(10 * nt, 200)
xa = np.unique(
    np.r_[np.linspace(0.01, 6.0, nx), xp]
)

nf = float(md["mean_field"]["n"])
uh0 = md["mean_field"]["uh"]

pv = eu.MFPars(
    v=vv,
    n=nf,
    uh=uh0,
)

# Zero-temperature reference
sz = eu.solve_eq(
    bd,
    pv,
    t=0.0,
    d=eq["initial"]["d"],
    m=eq["initial"]["m"],
    **eq["solve"],
)

dz = abs(sz.d)

# Critical temperature for this V
cp = {
    **eq["critical"],
    "prog": False,
}

tv = eu.critical_temperature(
    bd,
    pv,
    **cp,
)

t2 = 0.1 * tv

# Initial state at the cold-bath temperature
se = eu.solve_eq(
    bd,
    pv,
    t=t2,
    d=max(dz, 1.0e-8),
    m=sz.m,
    **eq["solve"],
)

nq = se.n.copy()
d = se.d
m = se.m

# Finite seed allows an unstable normal solution to leave d = 0
ds = max(0.02 * dz, 1.0e-3)

# The spectrum is centered at V/2
bp = {
    **bpars,
    "mu": 0.5 * vv,
}

da = np.full(xa.size, np.nan)
ma = np.full(xa.size, np.nan)
la = np.full(xa.size, np.nan)
era = np.full(xa.size, np.nan)
oka = np.zeros(xa.size, dtype=bool)

sp = [None] * xp.size

for i, x1 in tqdm(
    enumerate(xa),
    total=xa.size,
    desc="T1 scan",
):
    t1 = x1 * tv

    bs = (
        ej.gam_db(
            t=t1,
            name="bath 1",
            **bp,
        ),
        ej.gam_db(
            t=t2,
            name="bath 2",
            **bp,
        ),
    )

    so = ej.solve_open(
        bd,
        pv,
        nq,
        bs,
        d=d,
        m=m,
        **op["solve"],
    )

    ee = np.asarray(so.st.e)
    nn = np.asarray(so.n)
    de = ee[0] - ee[1]

    gp = gap_info(bd, so.st)

    da[i] = abs(so.d)
    ma[i] = so.m
    era[i] = so.err
    oka[i] = so.ok

    la[i] = vv * np.sum(
        np.asarray(bd.w)
        * (nn[1] - nn[0])
        / np.maximum(de, 1.0e-12)
    )

    jj = np.flatnonzero(
        np.isclose(xp, x1, rtol=0.0, atol=1.0e-12)
    )

    if jj.size:
        j = int(jj[0])
    
        uc = np.asarray(so.st.u)
        vc = np.asarray(so.st.v)
    
        na = uc**2 * nn[0] + vc**2 * nn[1]
        nb = vc**2 * nn[0] + uc**2 * nn[1]
    
        sp[j] = {
            "d": abs(float(so.d)),
            "m": float(so.m),
            "la": float(la[i]),
            "er": float(so.err),
            "ok": bool(so.ok),
            "eh": np.stack((
                np.asarray(so.st.eah),
                np.asarray(so.st.ebh),
            )),
            "ed": ee.copy(),
            "nq": nn.copy(),
            "no": np.stack((na, nb)),
            "u": uc.copy(),
            "v": vc.copy(),
            "gp": gp,
        }

    nq = so.n.copy()
    d = max(abs(so.d), ds)
    m = so.m

print(f"V = {vv:.4f}")
print(f"Delta_0 = {dz:.8f}")
print(f"T_c = {tv:.8f}")

T1 scan:   0%|          | 0/2004 [00:00<?, ?it/s]

V = 7.8000
Delta_0 = 1.00339244
T_c = 0.54960372


In [ ]:
kg = np.asarray(bd.k).reshape(
    bd.shape + (bd.dim,)
)

ea = np.asarray(bd.ea).reshape(bd.shape)
eb = np.asarray(bd.eb).reshape(bd.shape)

i0 = np.argmin(np.abs(kg[:, 0, 0]))
ip = np.argmin(kg[:, 0, 0])

p1 = np.column_stack((
    np.arange(i0, ip - 1, -1),
    np.full(i0 - ip + 1, i0),
))

p2 = np.column_stack((
    np.full(i0 - ip, ip),
    np.arange(i0 - 1, ip - 1, -1),
))

ii = np.arange(ip + 1, i0 + 1)
p3 = np.column_stack((ii, ii))

ij = np.vstack((p1, p2, p3))
kk = kg[ij[:, 0], ij[:, 1]]

dk = np.diff(kk, axis=0)
kd = np.r_[
    0.0,
    np.cumsum(np.linalg.norm(dk, axis=1)),
]

ix = len(p1) - 1
im = len(p1) + len(p2) - 1
kt = kd[[0, ix, im, -1]]

eap = ea[ij[:, 0], ij[:, 1]]
ebp = eb[ij[:, 0], ij[:, 1]]

In [ ]:
h = 2.15
fig, ax = plt.subplots(
    xp.size,
    1,
    sharex=True,
    sharey=True,
    figsize=(3.47412, h * 3.47412),
)

for j, a in tqdm(
    enumerate(ax),
    total=xp.size,
    desc="Plot",
):
    s = sp[j]

    eh = s["eh"].reshape(
        (2,) + bd.shape
    )
    ed = s["ed"].reshape(
        (2,) + bd.shape
    )

    ah = eh[0, ij[:, 0], ij[:, 1]] - 0.5 * vv
    bh = eh[1, ij[:, 0], ij[:, 1]] - 0.5 * vv

    al = ed[0, ij[:, 0], ij[:, 1]] - 0.5 * vv
    be = ed[1, ij[:, 0], ij[:, 1]] - 0.5 * vv

    a.plot(
        kd, eap,
        color=SET1_LIST[0],
        ls="--",
        lw=0.9,
        label=r"$\varepsilon_{a\mathbf{k}}$",
    )
    a.plot(
        kd, ebp,
        color=SET1_LIST[1],
        ls="--",
        lw=0.9,
        label=r"$\varepsilon_{b\mathbf{k}}$",
    )

    a.plot(
        kd, ah,
        color=SET1_LIST[0],
        lw=1.2,
        label=r"$\varepsilon^{\mathrm{H}}_{a\mathbf{k}}-V/2$",
    )
    a.plot(
        kd, bh,
        color=SET1_LIST[1],
        lw=1.2,
        label=r"$\varepsilon^{\mathrm{H}}_{b\mathbf{k}}-V/2$",
    )

    a.plot(
        kd, al,
        color=SET1_LIST[2],
        lw=1.3,
        label=r"$E_{\alpha\mathbf{k}}-V/2$",
    )
    a.plot(
        kd, be,
        color=SET1_LIST[3],
        lw=1.3,
        label=r"$E_{\beta\mathbf{k}}-V/2$",
    )

    a.axvline(
        kt[1],
        color="0.75",
        lw=0.7,
    )
    a.axvline(
        kt[2],
        color="0.75",
        lw=0.7,
    )

    tx = (
        rf"$T_1/T_c={xp[j]:g}$"
        "\n"
        rf"$\Delta/\Delta_0={s['d']/dz:.3f}$"
        "\n"
        rf"$\Lambda={s['la']:.3f}$"
    )

    a.text(
        0.02,
        0.07,
        tx,
        transform=a.transAxes,
        ha="left",
        va="bottom",
        fontsize=7,
        bbox={
            "facecolor": "white",
            "edgecolor": "0.8",
            "alpha": 0.85,
            "pad": 2,
        },
    )

    a.set_xlim(kd[0], kd[-1])
    a.set_ylabel(r"$\varepsilon_{\mathbf{k}}$")
    a.tick_params(direction="in")
    a.grid(alpha=0.3)

ax[-1].set_xticks(kt)
ax[-1].set_xticklabels([
    r"$\Gamma$",
    r"$X$",
    r"$M$",
    r"$\Gamma$",
])
ax[-1].set_xlabel(r"$\mathbf{k}$")

hh, ll = ax[0].get_legend_handles_labels()

fig.legend(
    hh,
    ll,
    bbox_to_anchor=(0, 0.93, 1, 0.08),
    loc="lower left",
    mode="expand",
    borderaxespad=0,
    ncol=3,
)

plt.tight_layout(rect=(0, 0, 1, 0.91))
plt.show()

In [ ]:
mu = 0.5 * vv
dn = max(dz, 1.0e-12)

# Find a common energy range for all five states
xl = np.inf
xr = -np.inf

for j in tqdm(
    range(xp.size),
    desc="Energy range",
):
    en = (
        np.asarray(sp[j]["ed"]) - mu
    ) / dn

    xl = min(xl, float(np.min(en)))
    xr = max(xr, float(np.max(en)))

pd = 0.05 * max(xr - xl, 1.0)
xe = np.linspace(xl - pd, xr + pd, 600)

In [ ]:
h = 1.9
fig, ax = plt.subplots(
    xp.size,
    1,
    sharex=True,
    sharey=True,
    figsize=(3.47412, h * 3.47412),
)

for j in tqdm(
    range(xp.size),
    desc="Distribution plots",
):
    s = sp[j]

    t1 = xp[j] * tv

    # Thermal distributions in normalized energy
    z1 = np.clip(xe * dn / t1, -700.0, 700.0)
    z2 = np.clip(xe * dn / t2, -700.0, 700.0)

    f1 = 1.0 / (np.exp(z1) + 1.0)
    f2 = 1.0 / (np.exp(z2) + 1.0)

    # NESS energies and quasiparticle occupations
    en = (
        np.asarray(s["ed"]) - mu
    ).reshape(2, -1) / dn

    nq = np.asarray(
        s["nq"]
    ).reshape(2, -1)

    a = ax[j]

    a.plot(
        xe,
        f1,
        color=SET1_LIST[0],
        lw=1.2,
        label=r"$f_{\mathrm{FD}}(E,T_1)$",
    )
    a.plot(
        xe,
        f2,
        color=SET1_LIST[1],
        lw=1.2,
        label=r"$f_{\mathrm{FD}}(E,T_2)$",
    )

    a.scatter(
        en[0],
        nq[0],
        color=SET1_LIST[2],
        s=6,
        alpha=0.55,
        edgecolors="none",
        label=r"$n_{\alpha\mathbf{k}}^{\mathrm{GGE}}$",
    )
    a.scatter(
        en[1],
        nq[1],
        color=SET1_LIST[3],
        s=6,
        alpha=0.55,
        edgecolors="none",
        label=r"$n_{\beta\mathbf{k}}^{\mathrm{GGE}}$",
    )

    a.axvline(
        0.0,
        color="0.55",
        ls="--",
        lw=0.7,
    )

    tx = (
        rf"$T_1/T_c={xp[j]:g}$"
        "\n"
        rf"$\Delta/\Delta_0={s['d']/dz:.3f}$"
        "\n"
        rf"$\Lambda={s['la']:.3f}$"
    )

    a.text(
        0.02,
        0.07,
        tx,
        transform=a.transAxes,
        ha="left",
        va="bottom",
        fontsize=7,
        bbox={
            "facecolor": "white",
            "edgecolor": "0.8",
            "alpha": 0.85,
            "pad": 2,
        },
    )

    a.set_xlim(xe[0], xe[-1])
    a.set_ylim(0.0, 1.0)
    a.set_ylabel(r"$n$")
    a.tick_params(direction="in")
    a.grid(alpha=0.3)

ax[-1].set_xlabel(
    r"$(E-V/2)/\Delta_0$"
)

hh, ll = ax[0].get_legend_handles_labels()

fig.legend(
    hh,
    ll,
    bbox_to_anchor=(0, 0.93, 1, 0.08),
    loc="lower left",
    mode="expand",
    borderaxespad=0,
    ncol=2,
)

plt.tight_layout(rect=(0, 0, 1, 0.91))
plt.show()

In [ ]:
x1 = 6.0
j = int(np.argmin(np.abs(xp - x1)))


t1 = x1 * tv
mu = 0.5 * vv

bp = {
    **bpars,
    "mu": mu,
}

bs = (
    ej.gam_db(
        t=t1,
        name="bath 1",
        **bp,
    ),
    ej.gam_db(
        t=t2,
        name="bath 2",
        **bp,
    ),
)

# Initial temperature between the two bath temperatures
b0 = 1.0 / (0.5 * (t1 + t2))

sf = et.solve_eff(
    bd,
    pv,
    bs,
    beta0=b0,
    d=sp[j]["d"],
    m=sp[j]["m"],
    mu=mu,
    dt=0.05,
    mix=(0.15, 0.15),
    tol=1.0e-6,
    nmax=20000,
    chk=10,
    mode=op["solve"].get("mode", "block"),
    block=op["solve"].get("block", 512),
    mf=False,
    prog=True,
)

print(f"Converged        = {sf.ok}")
print(f"Error            = {sf.err:.4e}")
print(f"beta_eff         = {sf.beta:.8f}")
print(f"T_eff            = {sf.t:.8f}")
print(f"T_eff/Tc         = {sf.t / tv:.8f}")
print(f"Power            = {sf.power:.4e}")
print(f"Delta_eff        = {sf.d:.8f}")
print(f"m_eff            = {sf.m:.8f}")

In [ ]:
en = (
    np.asarray(sp[j]["ed"]) - mu
) / dz

ef = (
    np.asarray(sf.st.e) - mu
) / dz

nn = np.asarray(sp[j]["nq"])

xl = min(float(np.min(en)), float(np.min(ef)))
xr = max(float(np.max(en)), float(np.max(ef)))
xe = np.linspace(xl, xr, 600)

f1 = 1.0 / (
    np.exp(np.clip(xe * dz / t1, -500.0, 500.0)) + 1.0
)
f2 = 1.0 / (
    np.exp(np.clip(xe * dz / t2, -500.0, 500.0)) + 1.0
)
fe = 1.0 / (
    np.exp(np.clip(sf.beta * xe * dz, -500.0, 500.0)) + 1.0
)

h = 0.68
fig, ax = plt.subplots(
    figsize=(3.47412, h * 3.47412)
)

ax.plot(
    xe, f1,
    color=SET1_LIST[0],
    ls="--",
    lw=1.0,
    label=r"$f_{\mathrm{FD}}(E,T_1)$",
)
ax.plot(
    xe, f2,
    color=SET1_LIST[1],
    ls="--",
    lw=1.0,
    label=r"$f_{\mathrm{FD}}(E,T_2)$",
)
ax.plot(
    xe, fe,
    color=SET1_LIST[4],
    lw=1.4,
    label=r"$f_{\mathrm{FD}}(E,T_{\mathrm{eff}})$",
)

ax.scatter(
    en[0],
    nn[0],
    color=SET1_LIST[2],
    s=7,
    alpha=0.55,
    edgecolors="none",
    label=r"$n_{\alpha\mathbf{k}}^{\mathrm{NESS}}$",
)
ax.scatter(
    en[1],
    nn[1],
    color=SET1_LIST[3],
    s=7,
    alpha=0.55,
    edgecolors="none",
    label=r"$n_{\beta\mathbf{k}}^{\mathrm{NESS}}$",
)

ax.axvline(
    0.0,
    color="0.65",
    ls=":",
    lw=0.7,
)

tx = (
    rf"$T_1/T_c={x1:g}$"
    "\n"
    rf"$T_2/T_c={t2/tv:.2f}$"
    "\n"
    rf"$T_\mathrm{{eff}}/T_c={sf.t/tv:.3f}$"
)

ax.text(
    0.02,
    0.06,
    tx,
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=7,
    bbox={
        "facecolor": "white",
        "edgecolor": "0.8",
        "alpha": 0.85,
        "pad": 2,
    },
)

ax.set_xlim(xe[0], xe[-1])
ax.set_ylim(0.0, 1.0)
ax.set_xlabel(r"$(E-V/2)/\Delta_0$")
ax.set_ylabel(r"$n$")
ax.tick_params(direction="in")
ax.grid(alpha=0.3)
ax.legend(
)

plt.tight_layout()
plt.show()

In [ ]:
# Single-bath equilibrium states
es = {**eq["solve"], "prog": False}

s1 = eu.solve_eq(
    bd, pv, t=t1,
    d=max(dz, 1.0e-3),
    m=sp[j]["m"],
    **es,
)

s2 = eu.solve_eq(
    bd, pv, t=t2,
    d=max(dz, 1.0e-3),
    m=sp[j]["m"],
    **es,
)

# NESS normal-state coordinate
eh = np.asarray(sp[j]["eh"])
nn = np.asarray(sp[j]["nq"])

xi = 0.5 * (eh[0] - eh[1])
xn = np.abs(xi) / dz

# Effective thermal-state coordinate
xif = 0.5 * (
    np.asarray(sf.st.eah)
    - np.asarray(sf.st.ebh)
)
xf = np.abs(xif) / dz

xr = max(float(np.max(xn)), float(np.max(xf)))
xe = np.linspace(0.0, xr, 600)

# Quasiparticle excitation energies
q1 = np.sqrt((xe * dz)**2 + s1.d**2)
q2 = np.sqrt((xe * dz)**2 + s2.d**2)
qf = np.sqrt((xe * dz)**2 + sf.d**2)

f1 = 1.0 / (np.exp(np.clip(q1 / t1, -500.0, 500.0)) + 1.0)
f2 = 1.0 / (np.exp(np.clip(q2 / t2, -500.0, 500.0)) + 1.0)
fe = 1.0 / (np.exp(np.clip(sf.beta * qf, -500.0, 500.0)) + 1.0)

In [ ]:
h = 0.68
fig, ax = plt.subplots(figsize=(3.47412, h * 3.47412))

ax.plot(xe, f1, color=SET1_LIST[0], ls="--", lw=1.0,
        label=r"$f_{\mathrm{FD}}(\xi,T_1)$")
ax.plot(xe, f2, color=SET1_LIST[1], ls="--", lw=1.0,
        label=r"$f_{\mathrm{FD}}(\xi,T_2)$")
ax.plot(xe, fe, color=SET1_LIST[4], lw=1.4,
        label=r"$f_{\mathrm{FD}}(\xi,T_{\mathrm{eff}})$")

ax.plot(xn, nn[0], color=SET1_LIST[2], alpha=0.5, label=r"$n_{\alpha\mathbf{k}}^{\mathrm{NESS}}$")
ax.scatter(xn, 1.0 - nn[1], color=SET1_LIST[3], s=7, alpha=0.5,
           edgecolors="none", label=r"$1-n_{\beta\mathbf{k}}^{\mathrm{NESS}}$")
 
ax.plot(xf, sf.n[0], alpha=0.5, linewidth=0.6,
           label=r"$n_{\alpha\mathbf{k}}^{\mathrm{th}}$")
ax.scatter(xf, 1.0 - sf.n[1], facecolors="none", edgecolors=SET1_LIST[4],
           marker="s", s=10, alpha=0.5, linewidths=0.6,
           label=r"$1-n_{\beta\mathbf{k}}^{\mathrm{th}}$")

tx = (
    rf"$T_1/T_c={x1:g}$"
    "\n"
    rf"$T_2/T_c={t2/tv:.2f}$"
    "\n"
    rf"$T_\mathrm{{eff}}/T_c={sf.t/tv:.3f}$"
)

ax.text(
    0.98, 0.96, tx, transform=ax.transAxes,
    ha="right", va="top", fontsize=7,
    bbox={"facecolor": "white", "edgecolor": "0.8",
          "alpha": 0.85, "pad": 2},
)

ax.set_xlim(xe[0], xe[-1])
ax.set_ylim(0.0, 0.52)
ax.set_xlabel(r"$|\xi_{\mathbf{k}}|/\Delta_0$")
ax.set_ylabel(r"$n_{\mathrm{ex}}$")
ax.tick_params(direction="in")
ax.grid(alpha=0.3)

ax.legend(
    bbox_to_anchor=(0, 1.02, 1, 0.2),
    loc="lower left", mode="expand",
    borderaxespad=0, ncol=3,
)

plt.tight_layout()
plt.show()

In [19]:
import fixed_baths as ss
import phase_map_plots as pm